# Generate Text Knowledge Base

The databases loaded in 5_download_dataset and cleaned in 6_cleaning are CSVs, RAGs prefer a text based knowledge base, so we are going to write structured `.txt` files to `data/3_txt_knowledge_base/`.

**Output layout:**
```
data/3_txt_knowledge_base/
  by_country/
    LGBT_Survey_ViolenceAndHarassment/
      e1_answer_by_Albania.txt
      e1_answer_by_Austria.txt
      ...
    LGBT_Survey_DailyLife/
      ...
  by_subset/
    LGBT_Survey_ViolenceAndHarassment/
      e1_answer_by_Lesbian.txt
      e1_answer_by_GayMan.txt
      ...
    LGBT_Survey_DailyLife/
      ...
```

**Each `.txt` file contains one line per country (or subset), sorted alphabetically.**

Line format — `by_subset` file (e.g. `e1_answer_by_Lesbian.txt`):
> `Question e1 — <question_label> | Lesbian responses in Austria: Yes: 33%, No: 51%, Don't know: 2%`

Line format — `by_country` file (e.g. `e1_answer_by_Austria.txt`):
> `Question e1 — <question_label> | Austria responses (Lesbian): Yes: 33%, No: 51%, Don't know: 2%`

**Prerequisite:** `6_cleaning.ipynb` — cleaned CSVs must exist in `data/2_cleaned/lgbt_EU/`.

## Setup

In [1]:
!pip install -r ../requirements.txt


[notice] A new release of pip available: 22.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import sys
from pathlib import Path

import pandas as pd

In [3]:
PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

CLEANED_LGBT_DIR  = PROJECT_ROOT / "data" / "2_cleaned" / "lgbt_EU"
TXT_KB_DIR        = PROJECT_ROOT / "data" / "3_txt_knowledge_base"

BY_COUNTRY_DIR = TXT_KB_DIR / "by_country"
BY_SUBSET_DIR  = TXT_KB_DIR / "by_subset"

BY_COUNTRY_DIR.mkdir(parents=True, exist_ok=True)
BY_SUBSET_DIR.mkdir(parents=True,  exist_ok=True)

print(f"Project root      : {PROJECT_ROOT}")
print(f"Cleaned LGBT dir  : {CLEANED_LGBT_DIR}")
print(f"Text KB output    : {TXT_KB_DIR}")
print(f"Cleaned dir exists: {'✅' if CLEANED_LGBT_DIR.exists() else '❌  Run 6_cleaning.ipynb first'}")

Project root      : C:\Users\RAZER\Desktop\portfolio-projects\1. RAG
Cleaned LGBT dir  : C:\Users\RAZER\Desktop\portfolio-projects\1. RAG\data\2_cleaned\lgbt_EU
Text KB output    : C:\Users\RAZER\Desktop\portfolio-projects\1. RAG\data\3_txt_knowledge_base
Cleaned dir exists: ✅


### Get cleaned CSV files

In [4]:
cleaned_files = sorted(CLEANED_LGBT_DIR.glob("*_cleaned.csv"))

print(f"Cleaned LGBT CSV files found: {len(cleaned_files)}")
for f in cleaned_files:
    print(f"  {f.name}")

Cleaned LGBT CSV files found: 5
  LGBT_Survey_DailyLife_cleaned.csv
  LGBT_Survey_Discrimination_cleaned.csv
  LGBT_Survey_RightsAwareness_cleaned.csv
  LGBT_Survey_TransgenderSpecificQuestions_cleaned.csv
  LGBT_Survey_ViolenceAndHarassment_cleaned.csv


# EU LGBTQ Dataset

## Execution
### Helper functions

- `build_response_string`
  - Turns a group of answer rows into a compact percentage string:
  - `Yes: 33%, I do not have a same-sex partner: 12%, Don't know: 2%`

- `make_by_subset_line` / `make_by_country_line`
  - Format one full line for the `.txt` file.

- `write_txt_file`
  - Writes a list of lines to a `.txt` file, creating parent dirs as needed.

In [5]:
def build_response_string(group: pd.DataFrame) -> str:
    """
    Given a DataFrame of rows for one (question_code, country, subset) triple,
    return a string like: "Yes: 33%, No: 51%, Don't know: 2%"
    Rows without a percentage value are skipped.
    """
    parts = []
    for _, row in group.iterrows():
        pct = row.get("percentage")
        answer = str(row["answer"]).strip()
        if pd.notna(pct):
            parts.append(f"{answer}: {int(pct)}%")
        else:
            parts.append(answer)
    return ", ".join(parts)


def make_by_subset_line(
    question_code: str,
    question_label: str,
    subset: str,
    country: str,
    response_str: str,
) -> str:
    """
    Format a line for a by_subset file.
    e.g. 'Question e1 — Do you avoid... | Lesbian responses in Austria: Yes: 33%, No: 51%'
    """
    return (
        f"Question {question_code} — {question_label} "
        f"| {subset} responses in {country}: {response_str}"
    )


def make_by_country_line(
    question_code: str,
    question_label: str,
    country: str,
    subset: str,
    response_str: str,
) -> str:
    """
    Format a line for a by_country file.
    e.g. 'Question e1 — Do you avoid... | Austria responses (Lesbian): Yes: 33%, No: 51%'
    """
    return (
        f"Question {question_code} — {question_label} "
        f"| {country} responses ({subset}): {response_str}"
    )


def write_txt_file(path: Path, lines: list[str]) -> None:
    """Write lines to a .txt file, one line per entry."""
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines) + "\n")

### Inspect one file before generating

Sanity check — confirm columns, unique question codes, countries, and subsets.

In [6]:
sample_path = cleaned_files[0]
sample_df = pd.read_csv(sample_path, encoding="utf-8", low_memory=False)

print(f"File         : {sample_path.name}")
print(f"Shape        : {sample_df.shape}")
print(f"Columns      : {list(sample_df.columns)}")
print(f"Question codes ({sample_df['question_code'].nunique()}): "
      f"{sorted(sample_df['question_code'].unique())[:10]} ...")
print(f"Countries    ({sample_df['CountryCode'].nunique()}): "
      f"{sorted(sample_df['CountryCode'].unique())}")
print(f"Subsets      ({sample_df['subset'].nunique()}): "
      f"{sorted(sample_df['subset'].unique())}")
sample_df.head(4)

File         : LGBT_Survey_DailyLife_cleaned.csv
Shape        : (32836, 6)
Columns      : ['CountryCode', 'subset', 'question_code', 'question_label', 'answer', 'percentage']
Question codes (52): ['b1_a', 'b1_b', 'b1_c', 'b1_d', 'b1_e', 'b1_f', 'b1_g', 'b1_h', 'b1_i', 'b2_a'] ...
Countries    (28): ['Austria', 'Belgium', 'Bulgaria', 'Croatia', 'Cyprus', 'Czech Republic', 'Denmark', 'Estonia', 'Finland', 'France', 'Germany', 'Greece', 'Hungary', 'Ireland', 'Italy', 'Latvia', 'Lithuania', 'Luxembourg', 'Malta', 'Netherlands', 'Poland', 'Portugal', 'Romania', 'Slovakia', 'Slovenia', 'Spain', 'Sweden', 'United Kingdom']
Subsets      (5): ['Bisexual men', 'Bisexual women', 'Gay', 'Lesbian', 'Transgender']


,CountryCode,subset,question_code,question_label,answer,percentage
0,Austria,Lesbian,b1_a,"In your opinion, how widespread is offensive l...",Very widespread,8
1,Austria,Lesbian,b1_a,"In your opinion, how widespread is offensive l...",Fairly widespread,34
2,Austria,Lesbian,b1_a,"In your opinion, how widespread is offensive l...",Fairly rare,45
3,Austria,Lesbian,b1_a,"In your opinion, how widespread is offensive l...",Very rare,9


## Core function

For each cleaned CSV file:
1. Derive the subfolder name from the file stem (strip `_cleaned` suffix)
2. Group by `(question_code, subset, CountryCode)` to build response strings
3. For **by_subset**: group lines by `(question_code, subset)`, one line per country, sorted A→Z
4. For **by_country**: group lines by `(question_code, CountryCode)`, one line per subset, sorted A→Z
5. Write each group to its `.txt` file

In [ ]:
def generate_txt_files_for_survey(csv_path: Path, by_country_root: Path, by_subset_root: Path) -> dict:
    """
    Generate by_subset and by_country .txt files from one cleaned LGBT survey CSV.

    Returns a stats dict with file counts for reporting.
    """
    df = pd.read_csv(csv_path, encoding="utf-8", low_memory=False)

    # Derive subfolder name: 'LGBT_Survey_ViolenceAndHarassment_cleaned' → 'LGBT_Survey_ViolenceAndHarassment'
    subfolder_name = csv_path.stem.replace("_cleaned", "")

    out_by_subset  = by_subset_root  / subfolder_name
    out_by_country = by_country_root / subfolder_name
    out_by_subset.mkdir(parents=True,  exist_ok=True)
    out_by_country.mkdir(parents=True, exist_ok=True)

    # Pre-build a lookup: (question_code, country, subset) → response_string
    # Group by all three keys; within each group, rows are the individual answer options
    group_keys = ["question_code", "CountryCode", "subset"]
    response_lookup: dict[tuple, str] = {}
    label_lookup: dict[str, str] = {}   # question_code → question_label (first seen)

    for (qcode, country, subset), grp in df.groupby(group_keys, sort=False):
        response_lookup[(qcode, country, subset)] = build_response_string(grp)
        if qcode not in label_lookup:
            label_lookup[qcode] = str(grp["question_label"].iloc[0]).strip()

    question_codes = sorted(df["question_code"].unique())
    all_subsets    = sorted(df["subset"].unique())
    all_countries  = sorted(df["CountryCode"].unique())

    by_subset_count  = 0
    by_country_count = 0

    # --- by_subset: one file per (question_code, subset) ---
    # File: <qcode>_answer_by_<subset>.txt
    # Each line: one country, sorted A→Z
    for qcode in question_codes:
        label = label_lookup[qcode]
        for subset in all_subsets:
            lines = []
            for country in all_countries:  # already sorted
                response_str = response_lookup.get((qcode, country, subset))
                if response_str is None:
                    continue  # this country/subset combo doesn't exist for this question
                lines.append(make_by_subset_line(qcode, label, subset, country, response_str))

            if lines:
                # Sanitize subset name for use in filename (replace spaces)
                safe_subset = subset.replace(" ", "")
                out_path = out_by_subset / f"{qcode}_answer_by_{safe_subset}.txt"
                write_txt_file(out_path, lines)
                by_subset_count += 1

    # --- by_country: one file per (question_code, country) ---
    # File: <qcode>_answer_by_<country>.txt
    # Each line: one subset, sorted A→Z
    for qcode in question_codes:
        label = label_lookup[qcode]
        for country in all_countries:
            lines = []
            for subset in all_subsets:  # already sorted
                response_str = response_lookup.get((qcode, country, subset))
                if response_str is None:
                    continue
                lines.append(make_by_country_line(qcode, label, country, subset, response_str))

            if lines:
                safe_country = country.replace(" ", "")
                out_path = out_by_country / f"{qcode}_answer_by_{safe_country}.txt"
                write_txt_file(out_path, lines)
                by_country_count += 1

    return {
        "subfolder":        subfolder_name,
        "question_codes":   len(question_codes),
        "subsets":          len(all_subsets),
        "countries":        len(all_countries),
        "by_subset_files":  by_subset_count,
        "by_country_files": by_country_count,
    }

### Run generation across all cleaned files

In [11]:
all_stats = []

for csv_path in cleaned_files:
    print(f"Processing: {csv_path.name} ...")
    stats = generate_txt_files_for_survey(csv_path, BY_COUNTRY_DIR, BY_SUBSET_DIR)
    all_stats.append(stats)
    print(f"  ✅  {stats['question_codes']} question codes  |  "
          f"{stats['subsets']} subsets  |  "
          f"{stats['countries']} countries  →  "
          f"{stats['by_subset_files']} by_subset files, "
          f"{stats['by_country_files']} by_country files")

print("\nDone.")

Processing: LGBT_Survey_DailyLife_cleaned.csv ...
DEBUG: qcode:b1_a		country:Austria		subset:Lesbian
DEBUG: qcode:b1_a		country:Austria		subset:Gay
DEBUG: qcode:b1_a		country:Austria		subset:Bisexual women
DEBUG: qcode:b1_a		country:Austria		subset:Bisexual men
DEBUG: qcode:b1_a		country:Austria		subset:Transgender
DEBUG: qcode:b1_a		country:Belgium		subset:Lesbian
DEBUG: qcode:b1_a		country:Belgium		subset:Gay
DEBUG: qcode:b1_a		country:Belgium		subset:Bisexual women
DEBUG: qcode:b1_a		country:Belgium		subset:Bisexual men
DEBUG: qcode:b1_a		country:Belgium		subset:Transgender
DEBUG: qcode:b1_a		country:Bulgaria		subset:Lesbian
DEBUG: qcode:b1_a		country:Bulgaria		subset:Gay
DEBUG: qcode:b1_a		country:Bulgaria		subset:Bisexual women
DEBUG: qcode:b1_a		country:Bulgaria		subset:Bisexual men
DEBUG: qcode:b1_a		country:Bulgaria		subset:Transgender
DEBUG: qcode:b1_a		country:Cyprus		subset:Lesbian
DEBUG: qcode:b1_a		country:Cyprus		subset:Gay
DEBUG: qcode:b1_a		country:Cyprus		subset:Transg

## Summary

In [12]:
total_by_subset  = sum(s["by_subset_files"]  for s in all_stats)
total_by_country = sum(s["by_country_files"] for s in all_stats)
total_files      = total_by_subset + total_by_country

print("=" * 65)
print("TEXT KNOWLEDGE BASE GENERATION SUMMARY")
print("=" * 65)

for s in all_stats:
    print(f"\n  {s['subfolder']}")
    print(f"    Question codes : {s['question_codes']}")
    print(f"    Subsets        : {s['subsets']}")
    print(f"    Countries      : {s['countries']}")
    print(f"    by_subset files: {s['by_subset_files']}")
    print(f"    by_country files:{s['by_country_files']}")

print(f"\n{'─'*65}")
print(f"  Total by_subset  files : {total_by_subset}")
print(f"  Total by_country files : {total_by_country}")
print(f"  Grand total            : {total_files} .txt files")
print(f"\n  Output: {TXT_KB_DIR}")

TEXT KNOWLEDGE BASE GENERATION SUMMARY

  LGBT_Survey_DailyLife
    Question codes : 52
    Subsets        : 5
    Countries      : 28
    by_subset files: 247
    by_country files:1418

  LGBT_Survey_Discrimination
    Question codes : 32
    Subsets        : 5
    Countries      : 28
    by_subset files: 144
    by_country files:877

  LGBT_Survey_RightsAwareness
    Question codes : 10
    Subsets        : 5
    Countries      : 28
    by_subset files: 50
    by_country files:280

  LGBT_Survey_TransgenderSpecificQuestions
    Question codes : 23
    Subsets        : 1
    Countries      : 27
    by_subset files: 23
    by_country files:571

  LGBT_Survey_ViolenceAndHarassment
    Question codes : 47
    Subsets        : 5
    Countries      : 28
    by_subset files: 235
    by_country files:1281

─────────────────────────────────────────────────────────────────
  Total by_subset  files : 699
  Total by_country files : 4427
  Grand total            : 5126 .txt files

  Output: C:\Us

### Spot-check output files

Read a sample `by_subset` and `by_country` file and print their contents.

In [13]:
# Pick the first subfolder and first .txt file in by_subset
sample_subset_files = sorted(BY_SUBSET_DIR.rglob("*.txt"))
sample_country_files = sorted(BY_COUNTRY_DIR.rglob("*.txt"))

if sample_subset_files:
    f = sample_subset_files[0]
    print(f"=== by_subset sample: {f.relative_to(TXT_KB_DIR)} ===")
    print(f.read_text(encoding="utf-8")[:1500])
    print()

=== by_subset sample: by_subset\LGBT_Survey_DailyLife\b1_a_answer_by_Bisexualmen.txt ===
Question b1_a — In your opinion, how widespread is offensive language about lesbian, gay, bisexual and/or transgender people by politicians in the country where you live? | Bisexual men responses in Austria: Very widespread: 5%, Fairly widespread: 20%, Fairly rare: 47%, Very rare: 20%, Don`t know: 9%
Question b1_a — In your opinion, how widespread is offensive language about lesbian, gay, bisexual and/or transgender people by politicians in the country where you live? | Bisexual men responses in Belgium: Very widespread: 1%, Fairly widespread: 9%, Fairly rare: 43%, Very rare: 42%, Don`t know: 4%
Question b1_a — In your opinion, how widespread is offensive language about lesbian, gay, bisexual and/or transgender people by politicians in the country where you live? | Bisexual men responses in Bulgaria: Very widespread: 38%, Fairly widespread: 26%, Fairly rare: 24%, Very rare: 8%, Don`t know: 5%
Quest

In [14]:
if sample_country_files:
    f = sample_country_files[0]
    print(f"=== by_country sample: {f.relative_to(TXT_KB_DIR)} ===")
    print(f.read_text(encoding="utf-8")[:1500])

=== by_country sample: by_country\LGBT_Survey_DailyLife\b1_a_answer_by_Austria.txt ===
Question b1_a — In your opinion, how widespread is offensive language about lesbian, gay, bisexual and/or transgender people by politicians in the country where you live? | Austria responses (Bisexual men): Very widespread: 5%, Fairly widespread: 20%, Fairly rare: 47%, Very rare: 20%, Don`t know: 9%
Question b1_a — In your opinion, how widespread is offensive language about lesbian, gay, bisexual and/or transgender people by politicians in the country where you live? | Austria responses (Bisexual women): Very widespread: 9%, Fairly widespread: 29%, Fairly rare: 48%, Very rare: 9%, Don`t know: 6%
Question b1_a — In your opinion, how widespread is offensive language about lesbian, gay, bisexual and/or transgender people by politicians in the country where you live? | Austria responses (Gay): Very widespread: 4%, Fairly widespread: 21%, Fairly rare: 52%, Very rare: 20%, Don`t know: 4%
Question b1_a — In

In [15]:
# Verify folder structure matches expected layout
print("Output folder structure:")
for top in sorted(TXT_KB_DIR.iterdir()):
    subfolders = sorted(top.iterdir())
    for sub in subfolders:
        txt_count = len(list(sub.glob("*.txt")))
        print(f"  {top.name}/{sub.name:<50}  {txt_count:>5} files")

Output folder structure:
  by_country/LGBT_Survey_DailyLife                                1418 files
  by_country/LGBT_Survey_Discrimination                            877 files
  by_country/LGBT_Survey_RightsAwareness                           280 files
  by_country/LGBT_Survey_TransgenderSpecificQuestions              571 files
  by_country/LGBT_Survey_ViolenceAndHarassment                    1281 files
  by_subset/LGBT_Survey_DailyLife                                 247 files
  by_subset/LGBT_Survey_Discrimination                            144 files
  by_subset/LGBT_Survey_RightsAwareness                            50 files
  by_subset/LGBT_Survey_TransgenderSpecificQuestions               23 files
  by_subset/LGBT_Survey_ViolenceAndHarassment                     235 files


# HIV AIDS Dataset
see `6_clustering` for more info, but there are 2 formats/schemas for data:
1. Schema A (no_of_*)
  - Columns: Country, Year, Count_median, Count_min, Count_max, WHO Region
  - Use: Count_median as the value
2. Schema B (art_*)
  - Columns: Country, Reported # of people receiving ART, Estimated number of people living with HIV_median,Estimated number of people living with HIV_min,	...
  
  - 👉 No Year, multiple columns = different indicators

# UNICEF Immunzation Dataset